# Stage 10 — Strong Constraints & Hallucination Suppression

**贯穿式 notebook**：每完成一个小阶段追加对应 C* 单元。

- **C0**：环境 / bootstrap / `medical_abbrev.json`
- **C0.5**：全量资源自检（chroma / chunks / BM25）+ 可选 Ollama 探活

> 验证策略：默认 **全量** `from_mode("full")`；单元与陷阱用 fixture（后续阶段）。

## C0：环境初始化 + bootstrap + 缩写表

In [1]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
STAGE10 = CWD.parent if CWD.name == "notebooks" else CWD
SRC = STAGE10 / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from bootstrap import bootstrap_paths
from config import DEFAULT_CONFIG
from resources import load_medical_abbrev

paths = bootstrap_paths(STAGE10)
abbrevs = load_medical_abbrev(STAGE10)

print("stage10:", paths["stage10"])
print("root:", paths["root"])
print("retrieval_mode:", DEFAULT_CONFIG.retrieval_mode)
print("max_retries:", DEFAULT_CONFIG.max_retries)
print("ref_strictness:", DEFAULT_CONFIG.ref_strictness)
print("refusal_en:", DEFAULT_CONFIG.refusal_en)
print("abbrev_count:", len(abbrevs))
for k in ("MI", "AF", "HF", "T2DM", "TAVR"):
    print(f"  {k} -> {abbrevs[k]}")

stage10: D:\谷歌\10 强约束规则开发与幻觉抑制
root: D:\谷歌
retrieval_mode: full
max_retries: 1
ref_strictness: relaxed
refusal_en: Based on the provided literature, this question cannot be answered.
abbrev_count: 35
  MI -> myocardial infarction
  AF -> atrial fibrillation
  HF -> heart failure
  T2DM -> type 2 diabetes mellitus
  TAVR -> transcatheter aortic valve replacement


## C0.5：全量资源自检 + 可选 Ollama

缺 chroma / chunks / BM25 分片时 `ready=False`（阶段 4–5 live 会失败）。  
Ollama 探活失败不阻断阶段 0（仅影响后续 live）。

In [2]:
import json
from resources import check_full_corpus_resources, probe_ollama

PROBE_OLLAMA = True  # set False if Ollama is offline during skeleton work

report = check_full_corpus_resources(STAGE10)
print(json.dumps({k: v for k, v in report.items() if k != "bm25_manifest"}, indent=2, ensure_ascii=False))
print("bm25_manifest:", report.get("bm25_manifest"))
assert report["ready"], (
    "Full corpus resources missing. Check chroma_db_full / oa_comm_chunks.jsonl / bm25_full. "
    f"status={report['status']}"
)
print("\nFull corpus READY.")

if PROBE_OLLAMA:
    ollama = probe_ollama()
    print("ollama:", json.dumps(ollama, indent=2, ensure_ascii=False))
else:
    print("ollama: skipped")

{
  "mode": "full",
  "chunks_path": "D:\\谷歌\\09 生成答案评估，缓存策略与批量处理\\data\\oa_comm_chunks.jsonl",
  "chunks_exists": true,
  "chunks_size_gb": 9.12,
  "chroma_persist_dir": "D:\\谷歌\\04 向量化与索引构建\\data\\chroma_db_full",
  "chroma_exists": true,
  "collection": "pmc_oa_comm_full",
  "bm25_cache_dir": "D:\\谷歌\\09 生成答案评估，缓存策略与批量处理\\data\\bm25_full",
  "bm25_ok": true,
  "slim_path": "D:\\谷歌\\06 检索系统开发第二部分\\data\\oa_comm_slim.jsonl",
  "slim_exists": true,
  "ready": true,
  "status": "ready"
}
bm25_manifest: {'format': 'bm25_sharded_v1', 'status': 'completed', 'num_shards': 62, 'total_chunks': 6107296}

Full corpus READY.
ollama: {
  "ok": true,
  "base_url": "http://127.0.0.1:11434",
  "model_requested": "deepseek-r1:7b",
  "model_available": true,
  "models_preview": [
    "deepseek-r1:7b"
  ]
}


## C1：ConstraintPromptBundle — 四层强约束（考前纪律手册）

> 本单元只展示约束文本，**不调用 LLM**。后续阶段 4 将通过 `append_to()` 叠加到 08 各步 system。

In [3]:
from constraint_prompts import default_constraint_bundle

bundle = default_constraint_bundle()

print("=== Four layers ===\n")
for name, text in bundle.layer_dict().items():
    print(f"--- {name} ---")
    print(text)
    print()

print("=== Full constraint block (as_system_prompt) ===\n")
full = bundle.as_system_prompt()
print(full)
print(f"\n[chars={len(full)}]")

# Simulate 08 answer_generator system + append (scheme A)
stage07_draft_system = (
    "You are a cautious medical assistant. "
    "Generate answers strictly grounded in provided evidence."
)
merged = bundle.append_to(stage07_draft_system)
print("\n=== append_to(stage07 draft system) preview ===\n")
print(merged[:400], "...\n[truncated]")
print(f"starts_with_stage07={merged.startswith(stage07_draft_system)}")
print(f"contains_refusal={bundle.refusal_en in merged}")

=== Four layers ===

--- knowledge_boundary ---
KNOWLEDGE BOUNDARY:
- Answer ONLY from the Retrieved Context / provided literature chunks.
- Do NOT use external knowledge, training data, or assumptions.
- If the question cannot be answered from the provided literature, you MUST respond with exactly this sentence (English canonical):
  "Based on the provided literature, this question cannot be answered."
- Chinese equivalent (if responding in Chinese): 根据现有文献无法回答此问题
- When refusing, output ONLY the refusal sentence (no extra advice, no speculative treatment suggestions).

--- citation_rules ---
CITATION RULES:
- Each context chunk is labeled with a temporary ID such as [1], [2], [3].
- Use ONLY citation IDs that appear in the provided context (canonical [n]).
- Do NOT cite [k] outside the assigned range (e.g. never cite [99] if only [1]–[5] were provided).
- Place citation markers [n] next to key factual claims supported by that chunk.
- Chinese alias [文献n] is acceptable but [n] is pref

### C1 输出解读

C1 **不调用 LLM**，只是把「考场纪律手册」打印出来。后续阶段 4 会用 `append_to()` 把它接到 08 的 system 后面。

#### 1）`=== Four layers ===` —— 四层纪律分别管什么

| 层名 | 对应任务书 | 通俗理解 |
|------|-----------|----------|
| `knowledge_boundary` | a 知识库边界 | 只能根据发下来的文献答；不够就写固定拒答句，不许瞎编建议 |
| `citation_rules` | b 引用来源 | 只能用 `[1][2]…` 这类已发编号，不许引用 `[99]` |
| `no_fabrication` | c 禁止编造 | 不许加文献里没有的剂量、结论、副作用 |
| `format_rules` | d 术语与格式 | 要分章节、缩写首次写全称、References 要有 title 等 |

四层合起来 = 考前完整纪律；阶段 3 的 `FormatChecker` 就是在考后**对照** `format_rules` 里写的那些要求。

---

#### 2）`=== Full constraint block ===` —— `as_system_prompt()` 成品

- 把四层拼成一块，开头有 `=== HARD CONSTRAINTS ===`。
- `[chars=1971]`：整段约 1971 字符，阶段 4 注入时会计入 system token。
- 这段可以**单独**作为 system，但本工程采用 **方案 A**：不替换 07 原文，而是追加。

---

#### 3）`=== append_to(stage07 draft system) preview ===` —— 方案 A 长什么样

```text
原 07 system（医学助手角色）
---
HARD CONSTRAINTS（四层纪律）
```

输出里的两行检查：

| 打印项 | 含义 |
|--------|------|
| `starts_with_stage07=True` | 07 原来的「谨慎医学助手」说明**还在**，没被覆盖 |
| `contains_refusal=True` | 纪律里已包含固定拒答句，OOD 题可被要求拒答 |

**此阶段效果**：模型还没被调用，但「答题前要多念一遍纪律」的文本已准备好。是否遵守要靠 C2/C3 考后检查 + 阶段 4 重试。

---

#### 和后续阶段的关系（一句话）

- **C1**：考前立规矩（prompt）
- **C2**：考后查引用编号
- **C3**：考后查卷面格式

## C2：CitationGuard — 编号分配与引用校验

> 纯规则演示，不调用 LLM。`assign_labels` 模拟 07 组装后为 chunk 加 `[1]…[k]`。

In [4]:
import json
from citation_guard import default_citation_guard

guard = default_citation_guard()

sample_chunks = [
    {"text": "Metformin improves cardiovascular outcomes in T2DM.", "chunk_id": "PMC_A"},
    {"text": "AMPK activation reduces myocardial fibrosis.", "chunk_id": "PMC_B"},
]

labeled = guard.assign_labels(sample_chunks)
print("=== assign_labels ===")
print(labeled.context_text)
print("valid_ids:", sorted(labeled.valid_ids))

good_answer = (
    "**Answer:** Metformin may benefit cardiovascular risk [1]. "
    "AMPK-related mechanisms are discussed [2].\n\n"
    "Evidence refs: [1] [2]"
)
bad_answer = "Answer citing only invalid source [99]."

for label, text in [("GOOD", good_answer), ("BAD", bad_answer)]:
    check = guard.validate(text, labeled.valid_ids)
    print(f"\n=== {label} validate ===")
    print(json.dumps(check.to_dict(), indent=2, ensure_ascii=False))

repaired, did = guard.retry_or_repair(bad_answer, guard.validate(bad_answer, labeled.valid_ids))
print("\n=== repair [99] ===")
print("repaired:", did)
print(repaired)
print("retry_hint:\n", guard.build_retry_hint(guard.validate(bad_answer, labeled.valid_ids)))

=== assign_labels ===
[1] Metformin improves cardiovascular outcomes in T2DM.

[2] AMPK activation reduces myocardial fibrosis.
valid_ids: [1, 2]

=== GOOD validate ===
{
  "ok": true,
  "extracted": [
    1,
    2
  ],
  "valid_ids": [
    1,
    2
  ],
  "invalid": [],
  "warnings": [],
  "issues": []
}

=== BAD validate ===
{
  "ok": false,
  "extracted": [
    99
  ],
  "valid_ids": [
    1,
    2
  ],
  "invalid": [
    99
  ],
  "warnings": [],
  "issues": [
    "Invalid citation ID(s) outside valid range [1, 2]: [99]"
  ]
}

=== repair [99] ===
repaired: True
Answer citing only invalid source .
retry_hint:
 Citation validation failed. Revise the answer with these rules:
- Remove or replace invalid citation IDs: [99]. Only use [1, 2].
- Invalid citation ID(s) outside valid range [1, 2]: [99]
- Place [n] markers next to supported claims.


### C2 输出解读

C2 演示 **CitationGuard**：先给文献贴临时编号 `[1][2]`，再检查答案里的引用是否合法。

#### 先认识 `validate` 返回的 JSON 字段

| 字段 | 含义 |
|------|------|
| `ok` | 引用是否过关（无 `issues` 即为 true） |
| `extracted` | 从答案里扫到的所有 `[n]` 编号 |
| `valid_ids` | 本次发卷允许的编号集合（如 `[1, 2]`） |
| `invalid` | 越界编号（如 `[99]` 不在允许范围内） |
| `issues` | 硬失败（目前主要是越界引用） |
| `warnings` | 软提醒（如「全文没出现任何 [n]」；默认 warn 不 fail） |

> 记忆：C2 主要抓 **引用编号越界**；没写引用默认只是黄灯（warn），不是红灯。

---

#### 1）`assign_labels` —— 发卷贴编号

```text
[1] Metformin improves cardiovascular outcomes in T2DM.
[2] AMPK activation reduces myocardial fibrosis.
valid_ids: [1, 2]
```

- 模拟 07 组装后、模型读 context 前：每段文献前有 `[n]`。
- `valid_ids` 与后面 08 `sources` 的 `index=1,2` **顺序一致**。
- 模型写「见 [1]」就是指第一篇文献。

---

#### 2）`GOOD validate` —— 合法引用

```json
"ok": true, "extracted": [1, 2], "invalid": []
```

- 答案引用了 `[1]` 和 `[2]`，都在允许范围内 → **通过**。
- 与 C3 区别：C2 不管有没有 `**Answer:**` 章节，只管编号对不对。

---

#### 3）`BAD validate` —— 非法引用 `[99]`

```json
"ok": false, "invalid": [99],
"issues": ["Invalid citation ID(s) outside valid range [1, 2]: [99]"]
```

- 只发了 `[1][2]`，答案却写 `[99]` → **硬失败**。
- 阶段 4 会：先 `retry`（附带 `retry_hint` 让模型重写），或 `retry_or_repair` 规则删掉 `[99]`。

---

#### 4）`repair [99]` —— 规则修补（不调用 LLM）

| 项 | 结果 |
|----|------|
| `repaired: True` | 代码改动了答案 |
| 修补后文本 | `Answer citing only invalid source .` —— `[99]` 被删掉 |

这是**保守修补**：只删非法标记，不帮模型重写内容。完整重写靠 `retry_hint` + 再次调用 LLM。

---

#### 5）`retry_hint` —— 给下次重试用的提示

示例内容要点：

- 列出非法编号 `[99]`
- 明确「只能用 `[1, 2]`」
- 提醒关键断言旁要加 `[n]`

阶段 4 会把这段附在修正 prompt 里，配合 `max_retries` 再生成一次。

---

#### C1 → C2 串联理解

1. **C1** 在 prompt 里告诉模型：「只能用已发编号」
2. **C2** 在答案生成后**机器核对**：真的只用了 `[1][2]` 吗？
3. 模型若仍写 `[99]` → C2 拦下 → 重试或规则删号

## C3：FormatChecker — 格式安检（章节 / 缩写 / 参考文献）

> 纯规则演示。`boundary_hit`（拒答固定句）时豁免三节要求。

In [5]:
import json
from resources import load_medical_abbrev
from format_checker import default_format_checker

fc = default_format_checker(load_medical_abbrev(STAGE10))

cases = {
    "GOOD": (
        "**Answer:**\n"
        "MI (myocardial infarction) may need reperfusion [1].\n\n"
        "**Evidence Summary:**\n- Context supports [1].\n\n"
        "Sources:\n"
        "[1] Homocysteine rhythms (doc_id=PMC520826, chunk_id=PMC520826, score=0.29)"
    ),
    "MISSING_SECTIONS": "**Answer:**\nMI (myocardial infarction) only.",
    "BARE_ABBREV": (
        "**Answer:**\nMI treatment discussed.\n\n"
        "**Evidence Summary:**\n- x\n\n"
        "Sources:\n[1] Paper (doc_id=PMC1)"
    ),
    "REFUSAL": fc.refusal_en,
}

for name, text in cases.items():
    r = fc.check(text)
    print(f"\n=== {name} ===")
    print(json.dumps(r.to_dict(), indent=2, ensure_ascii=False))

bad = fc.check(cases["MISSING_SECTIONS"])
patched, changed = fc.soft_patch(cases["MISSING_SECTIONS"], bad)
print("\n=== soft_patch (missing sections) ===")
print("changed:", changed)
print(patched[:300])


=== GOOD ===
{
  "ok": true,
  "issues": [],
  "warnings": [
    "Source [1] line has no journal/year (relaxed mode: warn only)."
  ],
  "score": 0.85,
  "boundary_hit": false,
  "sections_found": {
    "core_answer": true,
    "evidence_summary": true,
    "references": true
  },
  "abbrev_issues": []
}

=== MISSING_SECTIONS ===
{
  "ok": false,
  "issues": [
    "Missing required section: Evidence Summary / Evidence.",
    "Missing required section: References / Sources.",
    "Missing References/Sources section or sources list."
  ],
  "warnings": [],
  "score": 0.25,
  "boundary_hit": false,
  "sections_found": {
    "core_answer": true,
    "evidence_summary": false,
    "references": false
  },
  "abbrev_issues": []
}

=== BARE_ABBREV ===
{
  "ok": false,
  "issues": [
    "Abbreviation 'MI' first use should include full form (myocardial infarction), e.g. MI (myocardial infarction)."
  ],
  "warnings": [
    "Source [1] line has no journal/year (relaxed mode: warn only)."
  ],
 

### C3 输出解读

`FormatChecker` 像**卷面格式阅卷**：不判断医学内容对不对，只检查「章节齐不齐、缩写有没有写全称、参考文献字段够不够」。

#### 先认识 JSON 里每个字段

| 字段 | 含义 | 你怎么读 |
|------|------|----------|
| `ok` | **能不能过关** | `true` = 没有硬性问题（`issues` 为空）；`false` = 必须改 |
| `issues` | **硬失败项** | 出现就必须处理（重试或 `soft_patch`） |
| `warnings` | **软提醒** | 默认**不**导致 `ok=false`；relaxed 模式下缺 journal/year 常落在这里 |
| `score` | **格式分（0~1）** | 无 issues 且无 warnings → 1.0；只有 warnings → 0.85；有 issues 会扣分 |
| `boundary_hit` | **是否拒答** | 答案含固定拒答句时为 `true`，三节要求**自动豁免** |
| `sections_found` | **三节有没有** | `core_answer` / `evidence_summary` / `references` 各是否为 true |
| `abbrev_issues` | **缩写问题** | 首次出现缩写却没写全称时会列在这里 |

> 记忆：`issues` = 红灯；`warnings` = 黄灯；`ok` 只看红灯。

---

#### 1）`GOOD` —— 基本合格，但有一个黄灯

- 答案有 `**Answer:**`、`Evidence Summary`、`Sources:`，且写了 `MI (myocardial infarction)` → 三节和缩写都过。
- `ok: true`：格式**过关**。
- `warnings` 里「Source [1] line has no journal/year」：因为 08 的 `Sources:` 行通常只有 title + doc_id，**没有期刊和年份**。当前 `ref_strictness=relaxed`，所以只警告、不判 fail。
- `score: 0.85`：过关但有黄灯，所以不是满分 1.0。

---

#### 2）`MISSING_SECTIONS` —— 缺章节，红灯

- 只有 `**Answer:**` 一段，没有 Evidence Summary 和 Sources。
- `sections_found`：`evidence_summary=false`、`references=false`。
- `issues` 三条其实说同一件事：**缺两节 + 没有参考文献区**。
- `ok: false`，`score: 0.25`：格式不及格，阶段 4 会考虑重试或 `soft_patch`。

---

#### 3）`BARE_ABBREV` —— 章节齐了，但缩写没写全称

- 有完整三节，但正文只写 `MI treatment`，没有 `MI (myocardial infarction)`。
- `abbrev_issues` / `issues` 都会提示 MI 需要全称。
- `ok: false`：这是**硬失败**（和 C2 的非法引用类似，属于必须改的格式问题）。
- 同时仍有 journal/year 的 `warnings`（黄灯），所以 `score: 0.75`（一个问题扣 0.25）。

---

#### 4）`REFUSAL` —— 正确拒答，格式免检

- 答案只有固定句：`Based on the provided literature, this question cannot be answered.`
- `boundary_hit: true` → **不要求** Answer / Evidence / Sources 三节（否则「拒答反而格式不合格」就矛盾了）。
- `ok: true`，`score: 1.0`：拒答场景下的理想结果。
- 注意：`sections_found` 全 true 是**豁免后的标记**，不代表答案里真的有三节标题。

---

#### 5）`soft_patch` —— 缺章节时的「补骨架」

- 输入是 `MISSING_SECTIONS` 那条只有 Answer 的短文。
- `changed: True`：代码自动补上了：
  - `**Evidence Summary:**` + 占位句
  - `**Sources:**` + 占位句
- 这是**规则修补**，不是让 LLM 重写；目的是让卷面「至少有章节标题」，便于后续重试或人工查看。
- 缩写问题、引用问题**不会**被 soft_patch 自动修好。

---

#### 和 C2 的分工（一句话）

- **C2 CitationGuard**：引用编号 `[n]` 是否越界（如 `[99]`）。
- **C3 FormatChecker**：卷面格式（章节 / 缩写全称 / 参考文献字段）。

阶段 4 会把两者串在生成后：**先查引用，再查格式**。

## C4：ConstrainedGenerationPipeline — 端到端约束闭环

> **默认 fixture 演示**（不依赖 Ollama / 全量检索）：用假文献 + 脚本 LLM 展示 `constraint_checks` 闭环。  
> 将 `RUN_LIVE = True` 可切换全量 live（耗时数分钟，需 Ollama + 全量资源 READY）。

In [6]:
import json
from constrained_pipeline import ConstrainedGenerationPipeline
from config import Stage10Config

RUN_LIVE = False  # True → full corpus + Ollama (slow)

# --- Fixture demo (default): no retrieval / no Ollama ---
FIXTURE_CHUNKS = [
    {
        "chunk_id": "PMC_A",
        "doc_id": "PMC_A",
        "source_title": "Metformin cardiovascular study",
        "text": "Metformin improves cardiovascular outcomes in type 2 diabetes.",
        "final_score": 0.9,
    },
    {
        "chunk_id": "PMC_B",
        "doc_id": "PMC_B",
        "source_title": "AMPK and fibrosis",
        "text": "AMPK activation reduces myocardial fibrosis.",
        "final_score": 0.8,
    },
]

GOOD_FINAL = (
    "**Answer:** Metformin may benefit cardiovascular risk [1]. "
    "AMPK-related mechanisms are discussed [2].\n\n"
    "**Evidence Summary:**\n"
    "- Cardiovascular outcomes with metformin [1]\n"
    "- AMPK and fibrosis [2]\n"
)


class FixtureLLM:
    def generate(self, prompt, **kwargs):
        blob = (kwargs.get("system_prompt") or "") + prompt
        if "Draft:\n" not in blob and "CORRECTION REQUIRED" not in blob:
            return "Draft from evidence."
        return GOOD_FINAL


if RUN_LIVE:
    pipe = ConstrainedGenerationPipeline.from_mode(
        DEFAULT_CONFIG.retrieval_mode,
        config=DEFAULT_CONFIG,
        skip_evidence_eval=True,
        skip_critical_review=True,
        run_optional_eval=True,
    )
    query = "metformin cardiovascular effects"
    result = pipe.run(query)
else:
    from context_assembler import ContextAssembler

    class _NoRetrieval:
        def run(self, query: str):
            return {"query": query, "retrieval": {"fused": []}, "reranked": []}

    pipe = ConstrainedGenerationPipeline(
        retrieval_pipeline=_NoRetrieval(),
        context_assembler=ContextAssembler(tokenizer_name=None),
        llm_generator=FixtureLLM(),
        config=Stage10Config(max_retries=1),
        skip_evidence_eval=True,
        skip_critical_review=True,
        run_optional_eval=True,
    )
    query = "metformin cardiovascular effects (fixture)"
    result = pipe.run(query, fixture_chunks=FIXTURE_CHUNKS)

print("query:", result["query"])
print("retry_count:", result["retry_count"], "| repaired:", result["repaired"])
print("\n=== constraint_checks ===")
print(json.dumps(result["constraint_checks"], indent=2, ensure_ascii=False))
if result.get("optional_evaluation"):
    print("\n=== optional_evaluation (09 soft signal) ===")
    print(json.dumps(result["optional_evaluation"], indent=2, ensure_ascii=False))
print("\n=== labeled_context_preview ===")
print(result.get("labeled_context_preview", ""))
print("\n=== answer (first 600 chars) ===")
print((result["answer"] or "")[:600])

query: metformin cardiovascular effects (fixture)
retry_count: 0 | repaired: False

=== constraint_checks ===
{
  "citation": {
    "ok": true,
    "extracted": [
      1,
      2
    ],
    "valid_ids": [
      1,
      2
    ],
    "invalid": [],
    "warnings": [],
    "issues": []
  },
  "format": {
    "ok": true,
    "issues": [],
    "warnings": [
      "Source [1] missing journal.",
      "Source [1] missing year.",
      "Source [2] missing journal.",
      "Source [2] missing year."
    ],
    "score": 0.85,
    "boundary_hit": false,
    "sections_found": {
      "core_answer": true,
      "evidence_summary": true,
      "references": true
    },
    "abbrev_issues": []
  },
  "boundary_hit": false
}

=== optional_evaluation (09 soft signal) ===
{
  "hallucination_risk": 0.0,
  "hallucination_signals": []
}

=== labeled_context_preview ===
[1] Metformin improves cardiovascular outcomes in type 2 diabetes.

[2] AMPK activation reduces myocardial fibrosis.

=== answer (first 6

### C4 输出解读

C4 把 C1–C3 **串成一条流水线**：检索/组装 → 贴编号 → 注入纪律 → 生成 → 引用校验 → 格式校验 →（必要时）重试/修补。

本次运行：`RUN_LIVE=False`（fixture + 脚本 LLM），query = `metformin cardiovascular effects (fixture)`。

---

#### 1）顶部两行 —— 闭环是否触发重试/修补

```text
retry_count: 0 | repaired: False
```

| 字段 | 你的输出 | 含义 |
|------|----------|------|
| `retry_count` | `0` | 首次生成即通过 C2/C3，**没有**走 LLM 重写 |
| `repaired` | `False` | 没有规则删 `[99]` 或 `soft_patch` 补章节 |

→ 脚本答案一次写对，约束闭环「绿灯直过」。

---

#### 2）`constraint_checks.citation` —— C2 接入结果

```json
"ok": true, "extracted": [1, 2], "invalid": [], "issues": []
```

- 答案正文和 Evidence Summary 里引用了 `[1]`、`[2]`，都在 `valid_ids: [1, 2]` 内。
- 与 C2 的 GOOD 用例一致；08 后处理追加的 `Sources:` 块**不参与** citation 扫描（C2 设计如此）。

---

#### 3）`constraint_checks.format` —— C3 接入结果

```json
"ok": true, "score": 0.85,
"warnings": ["Source [1] missing journal.", "Source [1] missing year.", ...]
```

| 项 | 你的输出 | 对照 C3 |
|----|----------|---------|
| `sections_found` | 三节全 `true` | 有 `**Answer:**`、`**Evidence Summary:**`；`Sources:` 由后处理补上 |
| `abbrev_issues` | `[]` | 正文未出现需检测的缩写（MI/T2DM 等） |
| `ok: true` | 过关 | 与 C3 GOOD 相同：relaxed 下缺 journal/year 只在 `warnings` |
| `score: 0.85` | 有黄灯非满分 | 和 C3 GOOD 的 `0.85` 一致 |

> **记忆**：`format.ok` 只看 `issues`（红灯）；四条 journal/year 警告是黄灯，不触发重试。

---

#### 4）`boundary_hit: false` —— 本题是正常作答

- 不是拒答题，所以要求完整三节 + 合法引用。
- 若换成 OOD 题且输出固定拒答句，这里会变成 `true`，格式三节会自动豁免（见 C3 REFUSAL）。

---

#### 5）`optional_evaluation` —— 09 软信号（旁路对照）

```json
"hallucination_risk": 0.0, "hallucination_signals": []
```

- 09 启发式未扫到「100%」「completely safe」等措辞。
- **不决定**是否重试；硬门禁仍是上面的 `citation.ok` / `format.ok`。

---

#### 6）`labeled_context_preview` —— 发卷贴号已生效

```text
[1] Metformin improves cardiovascular outcomes in type 2 diabetes.
[2] AMPK activation reduces myocardial fibrosis.
```

- 07 组装后由 `assign_labels` 贴上 `[1][2]`，模型读到的 context 带编号。
- 答案里的 `[1]` → `Sources` 第一行 `Metformin cardiovascular study (doc_id=PMC_A, …)`。

---

#### 7）`answer` 末尾 —— 08 后处理叠加层

你的输出在模型正文之后自动加了：

- `Sources:` 两行（`index` 与 `[1][2]` 对齐，含 `doc_id` / `chunk_id` / `score`）
- `Medical disclaimer:` 免责声明

这是 08 `postprocess_answer` 的行为；C4 的校验在**后处理之后**跑，所以 `format.sections_found.references=true` 来自这段 `Sources:`。

---

#### 8）`RUN_LIVE` 切换时注意什么

| 模式 | 本次 | 切换后预期 |
|------|------|------------|
| `False`（默认） | 秒级、结果稳定 | 验证粘合逻辑 |
| `True` | 未跑 | 真检索 + Ollama；可能 `retry_count>0`、`score` 波动；单条数分钟级 |

C0.5 已显示全量资源与 Ollama **READY**，具备 live 条件；开发期仍建议 `max_retries=1`。

---

#### C1→C4 串联（一句话）

**C1 立规矩 → 生成时 append_to → C2/C3 考后验卷 → 本次一次通过（retry=0）→ 输出带 `constraint_checks` 的完整答卷。**

## C5：对抗用例批量跑测 + 指标表

> **默认 `--mock`**：脚本化答案 + fixture context，秒级出报告（不依赖 Ollama）。  
> `RUN_LIVE=True` 可走真 pipeline（耗时数分钟/条）。

In [7]:
import json
from pathlib import Path

from adversarial_eval import (
    default_cases_path,
    load_adversarial_cases,
    run_adversarial_eval,
    save_report,
)

RUN_LIVE = False  # True → ConstrainedGenerationPipeline.from_mode("full") + Ollama

cases = load_adversarial_cases(default_cases_path(STAGE10))
print(f"loaded {len(cases)} adversarial cases")
for c in cases:
    fx = "fixture" if c.use_fixture() else "live"
    print(f"  [{c.case_type:20}] {c.id} ({fx})")

if RUN_LIVE:
    from constrained_pipeline import ConstrainedGenerationPipeline

    pipe = ConstrainedGenerationPipeline.from_mode(
        DEFAULT_CONFIG.retrieval_mode,
        config=DEFAULT_CONFIG,
        skip_evidence_eval=True,
        skip_critical_review=True,
        run_optional_eval=True,
    )

    def run_case(case):
        if case.use_fixture():
            return pipe.run(case.query, fixture_chunks=case.fixture_chunks)
        return pipe.run(case.query)

else:
    # Mock runner (aligned with scripts/run_adversarial_eval.py --mock)
    from bootstrap import bootstrap_paths
    from config import Stage10Config
    from constrained_pipeline import ConstrainedGenerationPipeline
    from context_assembler import ContextAssembler

    bootstrap_paths(STAGE10)
    MOCK = {
        "ood_2025_fda": "Based on the provided literature, this question cannot be answered.",
        "induce_fabrication_side_effects": "Based on the provided literature, this question cannot be answered.",
        "terminology_tavr": (
            "**Answer:** TAVR (transcatheter aortic valve replacement) may help selected elderly patients [1].\n\n"
            "**Evidence Summary:**\n- TAVR outcomes [1]\n"
        ),
        "fake_citation_99": "**Answer:** Metformin safety profile [1].\n\n**Evidence Summary:**\n- Safety [1]\n",
        "normal_metformin": (
            "**Answer:** Metformin may benefit cardiovascular risk [1]. Mechanism [2].\n\n"
            "**Evidence Summary:**\n- CV outcomes [1]\n- AMPK [2]\n"
        ),
    }

    class _MockLLM:
        def __init__(self):
            self.case_id = "normal_metformin"

        def set_case(self, cid):
            self.case_id = cid

        def generate(self, prompt, **kwargs):
            blob = (kwargs.get("system_prompt") or "") + prompt
            if "Draft:\n" not in blob and "CORRECTION REQUIRED" not in blob:
                return "Draft."
            return MOCK.get(self.case_id, MOCK["normal_metformin"])

    class _NoRetrieval:
        def run(self, query):
            return {"query": query, "retrieval": {"fused": []}, "reranked": []}

    _llm = _MockLLM()
    _pipe = ConstrainedGenerationPipeline(
        retrieval_pipeline=_NoRetrieval(),
        context_assembler=ContextAssembler(tokenizer_name=None),
        llm_generator=_llm,
        config=Stage10Config(max_retries=1),
        skip_evidence_eval=True,
        skip_critical_review=True,
        run_optional_eval=True,
    )

    def run_case(case):
        _llm.set_case(case.id)
        chunks = case.fixture_chunks if case.fixture_chunks is not None else []
        return _pipe.run(case.query, fixture_chunks=chunks)

report = run_adversarial_eval(cases, run_case)
report["run_meta"] = {"mode": "live" if RUN_LIVE else "mock", "case_count": len(cases)}

# Metrics table
m = report["metrics"]
print("\n=== metrics ===")
for k in (
    "hallucination_rate",
    "refusal_hit_rate",
    "citation_accuracy",
    "format_compliance_rate",
    "terminology_compliance_rate",
    "soft_hallucination_risk_mean",
):
    print(f"  {k}: {m.get(k)}")

print("\n=== per-case scores ===")
rows = []
for item in report["cases"]:
    sc = item["score"]
    rows.append(
        {
            "id": sc["case_id"],
            "type": sc["case_type"],
            "hallucination_fail": sc["hallucination_fail"],
            "boundary_hit": sc["boundary_hit"],
            "citation_ok": sc["citation_ok"],
            "format_ok": sc["format_ok"],
        }
    )
print(json.dumps(rows, indent=2, ensure_ascii=False))

loaded 5 adversarial cases
  [ood                 ] ood_2025_fda (fixture)
  [induce_fabrication  ] induce_fabrication_side_effects (fixture)
  [terminology         ] terminology_tavr (fixture)
  [fake_citation       ] fake_citation_99 (fixture)
  [normal_control      ] normal_metformin (fixture)

=== metrics ===
  hallucination_rate: 0.0
  refusal_hit_rate: 1.0
  citation_accuracy: 1.0
  format_compliance_rate: 1.0
  terminology_compliance_rate: 1.0
  soft_hallucination_risk_mean: 0.0

=== per-case scores ===
[
  {
    "id": "ood_2025_fda",
    "type": "ood",
    "hallucination_fail": false,
    "boundary_hit": true,
    "citation_ok": true,
    "format_ok": true
  },
  {
    "id": "induce_fabrication_side_effects",
    "type": "induce_fabrication",
    "hallucination_fail": false,
    "boundary_hit": true,
    "citation_ok": true,
    "format_ok": true
  },
  {
    "id": "terminology_tavr",
    "type": "terminology",
    "hallucination_fail": false,
    "boundary_hit": false,
    "ci

## C6：导出报告 JSON + 关键结论

将 C5 的 `report` 写入 `outputs/samples/`，并摘录关键结论。

In [8]:
from adversarial_eval import save_report

suffix = "_full" if DEFAULT_CONFIG.retrieval_mode == "full" else ""
out_path = STAGE10 / "outputs" / "samples" / f"adversarial_eval_report{suffix}.json"
save_report(report, out_path)
print("saved:", out_path)

m = report["metrics"]
print("\n=== 关键结论（mock 演示预期） ===")
print(f"- 陷阱用例数（计入幻觉分母）: {m['hallucination_denominator']} / 总 {m['total_cases']}")
print(f"- 硬幻觉率: {m.get('hallucination_rate')} （越低越好）")
print(f"- OOD 拒答命中率: {m.get('refusal_hit_rate')} （OOD 用例应接近 1.0）")
print(f"- 引用准确率: {m.get('citation_accuracy')}")
print(f"- 格式合规率: {m.get('format_compliance_rate')}")
print(f"- 术语合规率: {m.get('terminology_compliance_rate')}")

fails = [c for c in report["cases"] if c["score"]["hallucination_fail"]]
if fails:
    print("\n未过关用例:")
    for item in fails:
        sc = item["score"]
        print(f"  - {sc['case_id']}: {sc['fail_reasons']}")
else:
    print("\n所有计入分母的陷阱用例均通过（mock 脚本答案下预期结果）。")

print("\nCLI 复现:")
print("  python scripts/run_adversarial_eval.py --mock")
print("  python scripts/run_adversarial_eval.py --mode live --retrieval-mode full --fixture-only")

saved: D:\谷歌\10 强约束规则开发与幻觉抑制\outputs\samples\adversarial_eval_report_full.json

=== 关键结论（mock 演示预期） ===
- 陷阱用例数（计入幻觉分母）: 3 / 总 5
- 硬幻觉率: 0.0 （越低越好）
- OOD 拒答命中率: 1.0 （OOD 用例应接近 1.0）
- 引用准确率: 1.0
- 格式合规率: 1.0
- 术语合规率: 1.0

所有计入分母的陷阱用例均通过（mock 脚本答案下预期结果）。

CLI 复现:
  python scripts/run_adversarial_eval.py --mock
  python scripts/run_adversarial_eval.py --mode live --retrieval-mode full --fixture-only


### C5/C6 输出解读

本次运行：`RUN_LIVE=False`（mock 脚本答案 + 5 条 fixture 用例）。C6 已写入 `outputs/samples/adversarial_eval_report_full.json`。

---

#### 1）用例清单 —— 5 条全是 fixture

```text
ood_2025_fda / induce_fabrication_side_effects / terminology_tavr /
fake_citation_99 / normal_metformin  (fixture)
```

- 陷阱题不跑全量检索，避免「碰巧检索到相关文献」导致 OOD 拒答失败。
- `normal_metformin` 本次也用 fixture；正式评测可单独 live 跑这一条。

---

#### 2）`=== metrics ===` —— 汇总指标（你的输出）

| 指标 | 你的值 | 怎么读 |
|------|--------|--------|
| `hallucination_rate` | **0.0** | 硬幻觉率：分母 3 条陷阱（OOD + 诱导 + 假引用），**0 条失败** |
| `refusal_hit_rate` | **1.0** | OOD 用例 `ood_2025_fda` 命中拒答句 |
| `citation_accuracy` | **1.0** | 所有提取到的 `[n]` 均合法 |
| `format_compliance_rate` | **1.0** | 5/5 条 `format_ok=true` |
| `terminology_compliance_rate` | **1.0** | `terminology_tavr` 写了 `TAVR (transcatheter aortic valve replacement)` |
| `soft_hallucination_risk_mean` | **0.0** | 09 软信号均值（mock 答案无绝对化措辞） |

> C6 打印：`陷阱用例数（计入幻觉分母）: 3 / 总 5` —— 正常对照 + 术语用例**不计入**幻觉分母。

---

#### 3）`per-case scores` —— 逐条对照

| id | type | hallucination_fail | boundary_hit | 你的结果说明 |
|----|------|-------------------|--------------|--------------|
| `ood_2025_fda` | ood | `false` ✓ | `true` | 空 context → mock 输出固定拒答句 → OOD 过关 |
| `induce_fabrication_side_effects` | induce_fabrication | `false` ✓ | `true` | 拒答而非编造「12% nausea」→ 诱导编造过关 |
| `terminology_tavr` | terminology | `false` | `false` | 正常作答 + 缩写带全称；**不看** hallucination_fail |
| `fake_citation_99` | fake_citation | `false` ✓ | `false` | mock 只引 `[1]`，无 `[99]` → 虚假引用过关 |
| `normal_metformin` | normal_control | `false` | `false` | 正常对照：引用/格式合规；**不进**幻觉分母 |

五条 `citation_ok=true`、`format_ok=true`，与 C4 单条 fixture 过关形态一致。

---

#### 4）mock 满分的含义（重要）

当前 **0.0 幻觉率 / 全 1.0** 是因为 mock 脚本**按用例 id 返回预设合规答案**，验证的是：

1. 用例 JSON 字段与计分逻辑是否对齐  
2. pipeline → `constraint_checks` → `score_adversarial_case` 链路是否通  

**不代表**真 Ollama 在全量 live 下也能 0 幻觉。切 `RUN_LIVE=True` 或 `python scripts/run_adversarial_eval.py --mode live` 才是模型真实表现。

---

#### 5）C6 导出与复现

```text
saved: .../outputs/samples/adversarial_eval_report_full.json
所有计入分母的陷阱用例均通过（mock 脚本答案下预期结果）。
```

CLI 等价命令：

```bash
python scripts/run_adversarial_eval.py --mock
python scripts/run_adversarial_eval.py --mode live --retrieval-mode full --fixture-only
```

---

#### 6）与 C4 的关系

- **C4**：单题端到端，看一条 `constraint_checks`。  
- **C5/C6**：同一 pipeline 批量跑 5 题，汇总成可交付报告 JSON。

下一步若 live：优先只对 `normal_metformin` 开检索，陷阱题继续 fixture；关注 `hallucination_rate` 是否上升、`retry_count` 是否 >0。